# HOVR-SG trên Google Colab với Visual Genome

Notebook này chạy toàn bộ workflow từ dữ liệu Visual Genome đến checkpoint `best.pt`: cài package, kiểm tra GPU, sinh ontology, convert unified JSONL, tạo train/val và SS/NS/SN/NN splits, train CLIP-HOVR-SG, validate checkpoint và evaluate.

> Visual Genome raw files cần được tải hợp pháp và đặt trên Google Drive trước khi chạy. Notebook không tự tải một archive lớn không rõ nguồn.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/hovr_sg')
VG_ROOT = Path('/content/drive/MyDrive/VisualGenome')
RAW_OBJECTS = VG_ROOT / 'objects.json'
RAW_RELATIONSHIPS = VG_ROOT / 'relationships.json'
RAW_IMAGE_DATA = VG_ROOT / 'image_data.json'
IMAGE_ROOT = VG_ROOT / 'images'
GENERATED_ROOT = VG_ROOT / 'generated'
ONTOLOGY_PATH = REPO_DIR / 'ontology/ontology_vg_v1.json'
ONTOLOGY_REPORT = GENERATED_ROOT / 'ontology_vg_v1_report.json'
UNIFIED_JSONL = GENERATED_ROOT / 'vg_unified.jsonl'
SPLITS_DIR = GENERATED_ROOT / 'splits'
RUN_DIR = VG_ROOT / 'runs/hovr_vg_release'
TRAIN_VAL_SOURCE = 'ss'  # strict zero-shot; use 'all' only for a non-strict pilot
VAL_RATIO = 0.10
SEED = 42

for path in [RAW_OBJECTS, RAW_RELATIONSHIPS, RAW_IMAGE_DATA, IMAGE_ROOT]:
    print(path, 'OK' if path.exists() else 'MISSING')
GENERATED_ROOT.mkdir(parents=True, exist_ok=True)
assert all(path.exists() for path in [RAW_OBJECTS, RAW_RELATIONSHIPS, RAW_IMAGE_DATA, IMAGE_ROOT]), 'Hãy đặt đủ VG raw files và images vào VG_ROOT.'

## Cấu trúc dữ liệu trên Google Drive

Đặt dữ liệu theo cấu trúc sau. `IMAGE_ROOT` phải là một thư mục chứa trực tiếp các file ảnh có tên `<image_id>.jpg`. Nếu bạn đang có đồng thời `VG_100K` và `VG_100K_2`, hãy gộp/copy chúng vào một thư mục `images` trước khi chạy converter.

In [ ]:
# Ví dụ cấu trúc mong đợi:
# /content/drive/MyDrive/VisualGenome/
# ├── image_data.json
# ├── objects.json
# ├── relationships.json
# └── images/
#     ├── 1.jpg
#     ├── 2.jpg
#     └── ...

image_metadata = json.loads(RAW_IMAGE_DATA.read_text(encoding='utf-8'))
missing_images = []
for item in image_metadata[:1000]:
    image_id = str(item['image_id'])
    if not (IMAGE_ROOT / f'{image_id}.jpg').exists():
        missing_images.append(image_id)
print('Checked first', min(1000, len(image_metadata)), 'images; missing:', len(missing_images))
if missing_images[:10]:
    print('Examples:', missing_images[:10])

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/sonbm-itealvn/hovr_sg.git', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'safetensors', 'scipy'], check=True)
print('Repository and dependencies ready:', REPO_DIR)

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Hãy chọn Runtime type = GPU trong Colab trước khi training CLIP.')
print('GPU:', torch.cuda.get_device_name(0))

## Sinh ontology Visual Genome

Ontology được sinh từ frequency và policy mẫu. Hãy xem coverage report sau cell này; nếu coverage quá thấp, chỉnh `configs/vg_ontology_policy.example.yaml` trước khi tiếp tục.

In [ ]:
policy_path = REPO_DIR / 'configs/vg_ontology_policy.example.yaml'
subprocess.run([
    sys.executable, 'tools/build_vg_ontology.py',
    '--objects', str(RAW_OBJECTS),
    '--relationships', str(RAW_RELATIONSHIPS),
    '--policy', str(policy_path),
    '--ontology-output', str(ONTOLOGY_PATH),
    '--report-output', str(ONTOLOGY_REPORT),
], cwd=REPO_DIR, check=True)
report = json.loads(ONTOLOGY_REPORT.read_text(encoding='utf-8'))
print('Object coverage:', report['object']['coverage'])
print('Predicate coverage:', report['predicate']['coverage'])
print('Selected object labels:', report['object']['selected_unique_labels'])
print('Selected predicates:', report['predicate']['selected_unique_labels'])

In [ ]:
# Convert raw Visual Genome annotations sang unified JSONL.
subprocess.run([
    sys.executable, 'tools/convert_visual_genome.py',
    '--images-root', str(IMAGE_ROOT),
    '--image-data', str(RAW_IMAGE_DATA),
    '--objects', str(RAW_OBJECTS),
    '--relationships', str(RAW_RELATIONSHIPS),
    '--ontology', str(ONTOLOGY_PATH),
    '--output', str(UNIFIED_JSONL),
], cwd=REPO_DIR, check=True)
print('Unified JSONL:', UNIFIED_JSONL, 'size:', UNIFIED_JSONL.stat().st_size)

In [ ]:
inspect_report = GENERATED_ROOT / 'unified_inspect.json'
subprocess.run([
    sys.executable, 'scripts/inspect_dataset.py',
    '--jsonl', str(UNIFIED_JSONL),
    '--ontology', str(ONTOLOGY_PATH),
    '--output', str(inspect_report),
], cwd=REPO_DIR, check=True)
print(inspect_report.read_text(encoding='utf-8')[:5000])

## Tạo train/val và novelty splits

Mặc định `TRAIN_VAL_SOURCE='ss'` để train chỉ trên seen-seen pool. Nếu dataset pilot quá nhỏ khiến SS rỗng, đặt biến thành `'all'`; lựa chọn đó không còn strict zero-shot.

In [ ]:
subprocess.run([
    sys.executable, 'tools/build_splits.py',
    '--input', str(UNIFIED_JSONL),
    '--ontology', str(ONTOLOGY_PATH),
    '--output-dir', str(SPLITS_DIR),
    '--object-novel-ratio', '0.20',
    '--relation-novel-ratio', '0.20',
    '--val-ratio', str(VAL_RATIO),
    '--train-val-source', TRAIN_VAL_SOURCE,
    '--seed', str(SEED),
], cwd=REPO_DIR, check=True)
manifest = json.loads((SPLITS_DIR / 'split_manifest.json').read_text(encoding='utf-8'))
print(json.dumps({k: manifest[k] for k in ['train_val_source', 'train_images', 'val_images', 'train_val_image_overlap', 'record_counts', 'image_counts']}, indent=2))
assert manifest['train_val_image_overlap'] == 0
assert (SPLITS_DIR / 'train.jsonl').exists() and (SPLITS_DIR / 'val.jsonl').exists()

In [ ]:
# Tạo config Colab release từ config mặc định.
import yaml
config = yaml.safe_load((REPO_DIR / 'configs/hovr_sg.yaml').read_text(encoding='utf-8'))
config['seed'] = SEED
config['model']['backbone'] = 'clip'
config['model']['backbone_name'] = 'openai/clip-vit-base-patch32'
config['training']['device'] = 'cuda'
config['training']['amp'] = True
config['training']['epochs'] = 10
config['training']['batch_size'] = 2
config['validation']['frequency'] = 1
config['validation']['selection_metric'] = 'object_mAP50_95'
config_path = REPO_DIR / 'configs/hovr_sg_colab_vg.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

## Training

Cell sau sẽ tải CLIP pretrained lần đầu, chạy Hungarian matching, union-region relation features, bốn stage training, validation mỗi epoch và ghi `last.pt`/`best.pt` vào Google Drive. Với Visual Genome đầy đủ, có thể cần tăng `epochs`, giảm `batch_size` hoặc dùng gradient accumulation ở phiên bản mở rộng.

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, 'scripts/train.py',
    '--config', str(config_path),
    '--train-jsonl', str(SPLITS_DIR / 'train.jsonl'),
    '--val-jsonl', str(SPLITS_DIR / 'val.jsonl'),
    '--ontology', str(ONTOLOGY_PATH),
    '--image-root', str(IMAGE_ROOT),
    '--output-dir', str(RUN_DIR),
    '--device', 'cuda',
], cwd=REPO_DIR, check=True)

In [ ]:
# Validate checkpoint release.
subprocess.run([
    sys.executable, 'scripts/validate_checkpoint.py',
    '--checkpoint', str(RUN_DIR / 'best.pt'),
    '--ontology', str(ONTOLOGY_PATH),
], cwd=REPO_DIR, check=True)

## Evaluation trên các novelty protocol

Cell này evaluate `ss` và `nn`; có thể thêm `ns` và `sn` vào danh sách.

In [ ]:
for protocol in ['ss', 'ns', 'sn', 'nn']:
    test_jsonl = SPLITS_DIR / f'{protocol}.jsonl'
    if not test_jsonl.exists() or test_jsonl.stat().st_size == 0:
        print('Skip empty protocol:', protocol)
        continue
    output = RUN_DIR / f'{protocol}_metrics.json'
    subprocess.run([
        sys.executable, 'scripts/evaluate.py',
        '--checkpoint', str(RUN_DIR / 'best.pt'),
        '--jsonl', str(test_jsonl),
        '--ontology', str(ONTOLOGY_PATH),
        '--image-root', str(IMAGE_ROOT),
        '--output', str(output),
        '--device', 'cuda',
    ], cwd=REPO_DIR, check=True)
    print(protocol, json.loads(output.read_text(encoding='utf-8'))['metrics'])

## Resume và tải artifact

Nếu Colab bị ngắt, chạy lại các cell chuẩn bị path/config rồi dùng `--resume` với `last.pt`. Sau khi validator trả về `valid: true`, `best.pt`, `best_manifest.json` và `training_summary.json` là các artifact cần lưu cùng ontology và split manifest.

In [ ]:
# Resume example; bỏ comment khi cần tiếp tục một run bị gián đoạn.
# subprocess.run([
#     sys.executable, 'scripts/train.py',
#     '--config', str(config_path),
#     '--train-jsonl', str(SPLITS_DIR / 'train.jsonl'),
#     '--val-jsonl', str(SPLITS_DIR / 'val.jsonl'),
#     '--ontology', str(ONTOLOGY_PATH),
#     '--image-root', str(IMAGE_ROOT),
#     '--output-dir', str(RUN_DIR),
#     '--resume', str(RUN_DIR / 'last.pt'),
#     '--device', 'cuda',
# ], cwd=REPO_DIR, check=True)

release_dir = VG_ROOT / 'release'
release_dir.mkdir(parents=True, exist_ok=True)
for artifact in ['best.pt', 'best_manifest.json', 'training_summary.json']:
    source = RUN_DIR / artifact
    if source.exists():
        shutil.copy2(source, release_dir / artifact)
shutil.copy2(ONTOLOGY_PATH, release_dir / ONTOLOGY_PATH.name)
shutil.copy2(SPLITS_DIR / 'split_manifest.json', release_dir / 'split_manifest.json')
print('Release artifacts:', sorted(path.name for path in release_dir.iterdir()))